# Excel Handler 엣지 케이스 테스트

이 노트북은 Excel 처리 시 발생할 수 있는 특수한 케이스들을 테스트합니다.

In [1]:
import pandas as pd
import numpy as np
from dh_tool.dataframe import DataFrame, Sheets
import tempfile
import os

## 1. 특수 문자가 포함된 시트 이름 테스트

In [2]:
# 특수 문자가 포함된 시트 이름으로 테스트
special_chars = ['Sheet*', 'Sheet/', 'Sheet\\', 'Sheet?', 'Sheet[', 'Sheet]', '']
data = pd.DataFrame({'A': [1, 2, 3]})
sheets = Sheets(data)

for sheet_name in special_chars:
    try:
        sheets.create_sheet(data, sheet_name)
        print(f"시트 생성 시도: {sheet_name} - 예상치 못한 성공")
    except Exception as e:
        print(f"시트 생성 시도: {sheet_name} - 예상된 에러: {str(e)}")

시트 생성 시도: Sheet* - 예상된 에러: Invalid character * found in sheet title
시트 생성 시도: Sheet/ - 예상된 에러: Invalid character / found in sheet title
시트 생성 시도: Sheet\ - 예상된 에러: Invalid character \ found in sheet title
시트 생성 시도: Sheet? - 예상된 에러: Invalid character ? found in sheet title
시트 생성 시도: Sheet[ - 예상된 에러: Invalid character [ found in sheet title
시트 생성 시도: Sheet] - 예상된 에러: Invalid character ] found in sheet title
시트 생성 시도:  - 예상된 에러: 'Worksheet  does not exist.'


## 2. 복잡한 데이터 타입 테스트

In [3]:
# 복잡한 데이터 타입을 포함한 데이터프레임
complex_data = {
    'nested_list': [[1, 2, 3], [4, 5, 6], [7, 8, 9]],
    'nested_dict': [{'a': 1}, {'b': 2}, {'c': 3}],
    'numpy_array': [np.array([1, 2, 3]), np.array([4, 5, 6]), np.array([7, 8, 9])],
    'mixed_types': [1, 'string', [1, 2]],
    'special_values': [np.inf, -np.inf, np.nan]
}

df_complex = pd.DataFrame(complex_data)
sheets = Sheets(df_complex)

# 임시 파일에 저장 시도
with tempfile.NamedTemporaryFile(suffix='.xlsx', delete=False) as tmp:
    try:
        sheets.save(tmp.name)
        print("복잡한 데이터 저장 성공")
        
        # 저장된 파일 다시 로드
        loaded_df = pd.read_excel(tmp.name)
        print("\n로드된 데이터:")
        display(loaded_df)
    except Exception as e:
        print(f"에러 발생: {str(e)}")
    finally:
        os.unlink(tmp.name)

복잡한 데이터 저장 성공

로드된 데이터:


,nested_list,nested_dict,numpy_array,mixed_types,special_values
0,"[1, 2, 3]",{'a': 1},[1 2 3],1,inf
1,"[4, 5, 6]",{'b': 2},[4 5 6],string,-inf
2,"[7, 8, 9]",{'c': 3},[7 8 9],"[1, 2]",NaN


In [4]:
sheets.save("test_tmp.xlsx")

## 3. 대용량 셀 데이터 테스트

In [5]:
# 매우 긴 문자열을 포함한 데이터
long_text = 'A' * 32767  # Excel 셀 최대 길이
too_long_text = 'A' * 32768  # Excel 셀 최대 길이 초과

long_data = {
    'normal_text': ['short text', 'medium text' * 10],
    'long_text': [long_text, 'normal text'],
    'too_long_text': [too_long_text, 'normal text']
}

df_long = pd.DataFrame(long_data)
sheets = Sheets(df_long)

with tempfile.NamedTemporaryFile(suffix='.xlsx', delete=False) as tmp:
    try:
        sheets.save(tmp.name)
        print("긴 텍스트 저장 성공")
    except Exception as e:
        print(f"에러 발생: {str(e)}")
    finally:
        os.unlink(tmp.name)

긴 텍스트 저장 성공


## 4. 시트 전환 및 데이터 일관성 테스트

In [6]:
# 여러 시트 간 전환하면서 데이터 일관성 확인
df1 = pd.DataFrame({'A': [1, 2, 3], 'B': ['a', 'b', 'c']})
df2 = pd.DataFrame({'X': [4, 5, 6], 'Y': ['d', 'e', 'f']})
df3 = pd.DataFrame({'P': [7, 8, 9], 'Q': ['g', 'h', 'i']})

sheets = Sheets(df1)
sheets.create_sheet(df2, "Sheet2")
sheets.create_sheet(df3, "Sheet3")

# 시트 전환하면서 데이터 확인
test_sequence = ["Sheet1", "Sheet2", "Sheet3", "Sheet1", "Sheet3", "Sheet2"]

for sheet_name in test_sequence:
    sheets.select_sheet(sheet_name)
    print(f"\n현재 시트: {sheet_name}")
    display(sheets.df)


현재 시트: Sheet1


,A,B
0,1,a
1,2,b
2,3,c



현재 시트: Sheet2


,X,Y
0,4,d
1,5,e
2,6,f



현재 시트: Sheet3


...

,A,B
0,1,a
1,2,b
2,3,c



현재 시트: Sheet3


,P,Q
0,7,g
1,8,h
2,9,i



현재 시트: Sheet2


,X,Y
0,4,d
1,5,e
2,6,f


## 5. 시트 조작 중 에러 복구 테스트

In [7]:
sheets = Sheets(pd.DataFrame({'A': [1, 2, 3]}))

# 존재하지 않는 시트 선택 시도
try:
    sheets.select_sheet("NonExistentSheet")
except Exception as e:
    print(f"예상된 에러 발생: {str(e)}")
    print(f"현재 활성 시트: {sheets.current_sheet}")

# 동일한 이름의 시트 생성 시도
sheets.create_sheet(pd.DataFrame({'B': [4, 5, 6]}), "Sheet1")
print(f"\n사용 가능한 시트 목록: {sheets.sheet_names}")

# 마지막 시트 삭제 시도
try:
    for sheet in sheets.sheet_names.copy():
        sheets.remove_sheet(sheet)
except Exception as e:
    print(f"\n마지막 시트 삭제 시 에러: {str(e)}")

예상된 에러 발생: 'Worksheet NonExistentSheet does not exist.'
현재 활성 시트: Sheet1
Sheet1 sheet은 이미 존재합니다, 이름을 바꿔주세요

사용 가능한 시트 목록: ['Sheet1']


In [8]:
sheets.sheet_names

[]

## 6. 대량의 시트 생성 및 삭제 테스트

In [9]:
sheets = Sheets(pd.DataFrame({'A': [1]}))

# 많은 수의 시트 생성
for i in range(100):  # Excel 제한: 255개 시트
    try:
        sheets.create_sheet(pd.DataFrame({f'Col_{i}': [i]}), f"Sheet_{i}")
        if i % 10 == 0:
            print(f"생성된 시트 수: {i+1}")
    except Exception as e:
        print(f"시트 생성 중단: {str(e)}")
        break

print(f"\n최종 시트 수: {len(sheets.sheet_names)}")

# 시트 무작위 삭제
import random
sheets_to_delete = list(sheets.sheet_names)[1:]  # 첫 번째 시트 제외
random.shuffle(sheets_to_delete)

for sheet in sheets_to_delete[:10]:  # 처음 10개만 삭제
    sheets.remove_sheet(sheet)
    print(f"시트 삭제: {sheet}")

print(f"\n남은 시트 수: {len(sheets.sheet_names)}")

생성된 시트 수: 1
생성된 시트 수: 11
생성된 시트 수: 21
생성된 시트 수: 31
생성된 시트 수: 41
생성된 시트 수: 51
생성된 시트 수: 61
생성된 시트 수: 71
생성된 시트 수: 81
생성된 시트 수: 91

최종 시트 수: 101
시트 삭제: Sheet_85
시트 삭제: Sheet_86
시트 삭제: Sheet_66
시트 삭제: Sheet_6
시트 삭제: Sheet_60
시트 삭제: Sheet_18
시트 삭제: Sheet_2
시트 삭제: Sheet_90
시트 삭제: Sheet_35
시트 삭제: Sheet_10

남은 시트 수: 91
